# Logistic Regression Baseline

In [25]:
from pathlib import Path
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)

In [26]:
DATA_DIR = Path("data/merged")
GENRE = "FPS"  # Change to "MIXED" for the mixed dataset.

train_df = pd.read_csv(DATA_DIR / f"{GENRE}_train.csv")
val_df = pd.read_csv(DATA_DIR / f"{GENRE}_val.csv")
test_df = pd.read_csv(DATA_DIR / f"{GENRE}_test.csv")


In [27]:
# define a logistic regression model
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
    )),
    ("classifier", LogisticRegression(
        C=1.2,
        max_iter=1000,
    ))
])

# Train without automatically rendering the fitted pipeline in Jupyter.
_ = model.fit(
    train_df["review"],
    train_df["voted_up"]
)

In [28]:
# evaluate model performance
def evaluate_model(model, data):
    y_true = data["voted_up"]
    y_pred = model.predict(data["review"])

    print(classification_report(
        y_true, y_pred, labels=[1,0], target_names=["recommended", "not recommend"],
        zero_division=0, digits=3
    ))

    print("Macro-F1:", f1_score(
        y_true, y_pred,
        average="macro",
        zero_division=0,
    ))
    
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred, labels=[1,0]))

In [29]:
evaluate_model(model, val_df)

               precision    recall  f1-score   support

  recommended      0.871     0.756     0.809       884
not recommend      0.827     0.913     0.868      1133

     accuracy                          0.844      2017
    macro avg      0.849     0.834     0.839      2017
 weighted avg      0.846     0.844     0.842      2017

Macro-F1: 0.8385101108625019
Confusion matrix:
[[ 668  216]
 [  99 1034]]


In [30]:
print("Test set results:")
evaluate_model(model, test_df)

Test set results:
               precision    recall  f1-score   support

  recommended      0.872     0.779     0.823       882
not recommend      0.841     0.911     0.875      1136

     accuracy                          0.853      2018
    macro avg      0.857     0.845     0.849      2018
 weighted avg      0.855     0.853     0.852      2018

Macro-F1: 0.8488244137253811
Confusion matrix:
[[ 687  195]
 [ 101 1035]]


## External tests

Run the cells below in this notebook's trained kernel. They evaluate all games in
`config.EXTERNAL_TEST_GAME_IDS` and the balanced 4,000-review IMDb movie test set.
Reuse the current trained model and its training-time text processing; no fitting
or vocabulary construction is performed here. IMDb uses sentiment labels
(0=negative, 1=positive), while Steam uses recommendation labels.


In [31]:
import importlib
import config

# Refresh dataset configuration without restarting the trained kernel.
EXTERNAL_TEST_GAME_IDS = importlib.reload(config).EXTERNAL_TEST_GAME_IDS

external_test_dfs = {}
for game, appid in EXTERNAL_TEST_GAME_IDS.items():
    external_df = pd.read_csv(
        Path("data/processed") / f"reviews_{game}_{appid}.csv",
        usecols=["review", "voted_up"],
        dtype={"review": "string", "voted_up": "boolean"},
    )
    external_df = external_df.dropna(subset=["review", "voted_up"]).copy()
    external_df["voted_up"] = external_df["voted_up"].astype("int64")
    external_test_dfs[game] = external_df
    print(f"{game}: {len(external_df)} external test reviews")

# IMDb sentiment labels already use 0=negative and 1=positive.
imdb_test_df = pd.read_csv(
    Path("data/processed/reviews_IMDB_test.csv"),
    usecols=["review", "voted_up"],
    dtype={"review": "string", "voted_up": "int64"},
)
imdb_test_df = imdb_test_df.dropna(subset=["review", "voted_up"]).copy()
external_test_dfs["IMDB"] = imdb_test_df
print(f"IMDB: {len(imdb_test_df)} external test reviews")


WARDOGS: 1826 external test reviews
SQUAD: 1464 external test reviews
CYBERPUNK_2077: 1558 external test reviews
PERSONA_5_ROYAL: 1527 external test reviews
BALDURS_GATE_3: 1598 external test reviews
DOTA2: 967 external test reviews
HOLLOW_KNIGHT_SILKSONG: 1516 external test reviews
STARDEW_VALLEY: 1445 external test reviews
ELDEN_RING: 1476 external test reviews
FORZA_HORIZON_6: 1532 external test reviews
AMONG_US: 1301 external test reviews
IMDB: 4000 external test reviews


In [32]:
from sklearn.metrics import accuracy_score

# Reuse the fitted pipeline: predict transforms text without fitting TF-IDF again.
external_results = []
for game, external_df in external_test_dfs.items():
    y_true = external_df["voted_up"]
    y_pred = model.predict(external_df["review"])
    print(f"\n{game}")
    print(classification_report(
        y_true, y_pred, labels=[1, 0],
        target_names=(["positive", "negative"] if game == "IMDB"
                      else ["recommended", "not recommended"]),
        zero_division=0, digits=3,
    ))
    print("Confusion matrix (rows=true, columns=predicted; order=[1, 0]):")
    print(confusion_matrix(y_true, y_pred, labels=[1, 0]))
    external_results.append({
        "game": game,
        "samples": len(external_df),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    })

external_results = pd.DataFrame(external_results).set_index("game")
print("\nExternal test summary:")
print(external_results.to_string(float_format=lambda value: f"{value:.4f}"))



WARDOGS
                 precision    recall  f1-score   support

    recommended      0.935     0.600     0.731       883
not recommended      0.720     0.961     0.823       943

       accuracy                          0.786      1826
      macro avg      0.827     0.780     0.777      1826
   weighted avg      0.824     0.786     0.778      1826

Confusion matrix (rows=true, columns=predicted; order=[1, 0]):
[[530 353]
 [ 37 906]]

SQUAD
                 precision    recall  f1-score   support

    recommended      0.884     0.710     0.788       611
not recommended      0.818     0.933     0.872       853

       accuracy                          0.840      1464
      macro avg      0.851     0.822     0.830      1464
   weighted avg      0.846     0.840     0.837      1464

Confusion matrix (rows=true, columns=predicted; order=[1, 0]):
[[434 177]
 [ 57 796]]

CYBERPUNK_2077
                 precision    recall  f1-score   support

    recommended      0.849     0.700     0.767  